In [ ]:
import numpy as np

def relu(x):
    return np.maximum(x, 0)

def relu_derivative(x):
    return np.where(x > 0, 1, 0)

def f(W, x):
    return np.linalg.norm(relu(W @ x))**2

def gradient_W(W, x):
    relu_Wx = relu(W @ x)
    relu_derivative_Wx = relu_derivative(W @ x)
    return 2 * np.outer(relu_Wx * relu_derivative_Wx, x.T)

def gradient_x(W, x):
    relu_Wx = relu(W @ x)
    relu_derivative_Wx = relu_derivative(W @ x)
    return 2 * W.T @ (relu_Wx * relu_derivative_Wx)

W = np.random.randn(3, 3)
x = np.random.randn(3, 1)

print('W:', W)
print('x:', x)
print('f(W, x):', f(W, x))
print('gradient_W:', gradient_W(W, x))
print('gradient_x:', gradient_x(W, x))

W: [[ 1.60454146  0.97775232 -0.31819161]
 [-0.63201357  0.4546175   0.61305739]
 [-0.80610427  0.64738424 -3.25752685]]
x: [[-0.51966845]
 [-0.97971334]
 [ 0.26848986]]
f(W, x): 0.0022697963303915146
gradient_W: [[-0.         -0.          0.        ]
 [-0.04951648 -0.09335175  0.02558299]
 [-0.         -0.          0.        ]]
gradient_x: [[-0.06022126]
 [ 0.04331812]
 [ 0.05841503]]


In [14]:
import numpy as np
from urllib import request
import gzip
import pickle
import os
from sklearn.metrics import accuracy_score
from tqdm import tqdm

filename = [
    ["training_images", "train-images-idx3-ubyte.gz"],
    ["test_images", "t10k-images-idx3-ubyte.gz"],
    ["training_labels", "train-labels-idx1-ubyte.gz"],
    ["test_labels", "t10k-labels-idx1-ubyte.gz"],
]


def download_mnist():
    base_url = "https://ossci-datasets.s3.amazonaws.com/mnist/"
    for name in filename:
        if not os.path.exists(name[1]):
            print(f"Downloading {name[1]}...")
            request.urlretrieve(base_url + name[1], name[1])
    print("Download complete.")


def save_mnist():
    mnist = {}
    for name in filename[:2]:
        with gzip.open(name[1], "rb") as f:
            mnist[name[0]] = np.frombuffer(f.read(), np.uint8, offset=16).reshape(
                -1, 28 * 28
            )
    for name in filename[-2:]:
        with gzip.open(name[1], "rb") as f:
            mnist[name[0]] = np.frombuffer(f.read(), np.uint8, offset=8)
    with open("mnist.pkl", "wb") as f:
        pickle.dump(mnist, f)
    print("MNIST dataset saved as mnist.pkl.")


def load_mnist():
    if not os.path.exists("mnist.pkl"):
        download_mnist()
        save_mnist()
    with open("mnist.pkl", "rb") as f:
        mnist = pickle.load(f)
    return (
        mnist["training_images"],
        mnist["training_labels"],
        mnist["test_images"],
        mnist["test_labels"],
    )

X_train, y_train, X_test, y_test = load_mnist()
X_train, X_test = X_train / 255.0, X_test / 255.0

def one_hot_encode(y, num_classes=10):
    return np.eye(num_classes)[y]

y_train_one_hot = one_hot_encode(y_train)
y_test_one_hot = one_hot_encode(y_test)

np.random.seed(0)
W = np.random.randn(10, 784) * 0.01
b = np.zeros((10, 1))

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=0, keepdims=True))
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

def cross_entropy_loss(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-8), axis=1))
def compute_gradients(X, y_true, W, b):
    m = X.shape[0]
    Z = np.dot(W, X.T) + b
    A = softmax(Z)
    dZ = A - y_true.T 
    dW = (1/m) * np.dot(dZ, X)
    db = (1/m) * np.sum(dZ, axis=1, keepdims=True)
    return dW, db

def train(X, y, W, b, learning_rate=0.3, epochs=1000):
    for epoch in tqdm(range(epochs), desc="Training:"):
        dW, db = compute_gradients(X, y, W, b)
        W -= learning_rate * dW
        b -= learning_rate * db
        
        if epoch % 100 == 0:
            y_pred = softmax(np.dot(W, X.T) + b).T
            loss = cross_entropy_loss(y, y_pred)
            print(f"Epoch {epoch}: Loss = {loss:.4f}")
    return W, b

W, b = train(X_train, y_train_one_hot, W, b, learning_rate=0.5, epochs=1000)
y_pred_test = np.argmax(softmax(np.dot(W, X_test.T) + b), axis=0)
accuracy = accuracy_score(y_test, y_pred_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Training::   0%|          | 3/1000 [00:00<01:51,  8.97it/s]

Epoch 0: Loss = 1.8448


Training::  10%|█         | 102/1000 [00:10<01:37,  9.24it/s]

Epoch 100: Loss = 0.3919


Training::  20%|██        | 202/1000 [00:18<01:08, 11.57it/s]

...

Epoch 800: Loss = 0.2904


Training::  90%|█████████ | 902/1000 [01:21<00:08, 11.58it/s]

Epoch 900: Loss = 0.2870


Training:: 100%|██████████| 1000/1000 [01:34<00:00, 10.56it/s]

Test Accuracy: 92.15%
